# Reproducibility

This matters because debugging and comparison are impossible when runs drift for reasons you did not record. Focus on seeds, deterministic settings, and the metadata that belongs beside model weights.


In [ ]:
import random

import numpy as np
import torch

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.use_deterministic_algorithms(True)
print('deterministic algorithms enabled')


## Reset seeds and regenerate

In [ ]:
def make_sample():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    return {
        'python': random.random(),
        'numpy': float(np.random.rand()),
        'torch': torch.randn(3),
    }

sample_a = make_sample()
sample_b = make_sample()
print(sample_a)
assert sample_a['python'] == sample_b['python']
assert sample_a['numpy'] == sample_b['numpy']
assert torch.equal(sample_a['torch'], sample_b['torch'])

## Save metadata with checkpoints

A reproducible checkpoint should include more than model weights.

In [ ]:
checkpoint = {
    'seed': SEED,
    'torch_version': str(torch.__version__),
    'device_count': torch.cuda.device_count(),
    'deterministic_algorithms': True,
}
path = './output/reproducibility-metadata.pt'
torch.save(checkpoint, path)
reloaded = torch.load(path, map_location='cpu')
print(reloaded)
assert reloaded['seed'] == SEED